In [ ]:
#@title Configuration
ANTHROPIC_API_KEY = "" #@param {type:"string"}
OPENAI_API_KEY    = "" #@param {type:"string"}

In [ ]:
#@title Setup
import os, sys, subprocess, time

ROOT = "/content/nanobot"
PY   = sys.executable

# Clone
if os.path.exists(ROOT):
    print("Repo already exists, pulling latest...")
    subprocess.run(["git", "-C", ROOT, "pull"], check=True, capture_output=True)
else:
    print("Cloning repo...")
    subprocess.run(["git", "clone", "https://github.com/aho-ui/ERP-Agent.git", ROOT], check=True, capture_output=True)

# Write .env
with open(f"{ROOT}/.env", "w") as f:
    f.write(f"ANTHROPIC_API_KEY={ANTHROPIC_API_KEY}\n")
    if OPENAI_API_KEY:
        f.write(f"OPENAI_API_KEY={OPENAI_API_KEY}\n")
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["OPENAI_API_KEY"]    = OPENAI_API_KEY

# Python deps
print("Installing Python dependencies...")
subprocess.run([PY, "-m", "pip", "install", "-r", f"{ROOT}/requirements.txt", "-q"], check=True)

# Node.js
print("Installing Node.js...")
subprocess.run("curl -fsSL https://deb.nodesource.com/setup_20.x | bash -", shell=True, capture_output=True)
subprocess.run(["apt-get", "install", "-y", "nodejs"], capture_output=True)

# Frontend deps
print("Installing frontend dependencies...")
subprocess.run(["npm", "ci", "-q"], check=True, cwd=f"{ROOT}/frontend")

# Demo data
print("Generating demo data...")
sys.path.insert(0, f"{ROOT}/generator")
from sqlite.main import run as generate_demo
generate_demo()

# Django
print("Running migrations...")
r = subprocess.run([PY, "manage.py", "migrate", "--no-input"], cwd=ROOT, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout); print(r.stderr)
    raise RuntimeError("Migration failed.")
subprocess.run([PY, "manage.py", "create_default_users"], cwd=ROOT, check=True)

# cloudflared
print("Downloading cloudflared...")
subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/usr/local/bin/cloudflared"])
subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"])

print("Setup complete.")

In [ ]:
#@title Run
import subprocess, threading, re, time, os, sys

ROOT = "/content/nanobot"

def _capture_tunnel_url(port):
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        match = re.search(r'https://[\w\-]+\.trycloudflare\.com', line)
        if match:
            return match.group(0)

# Start backend
print("Starting backend...")
subprocess.Popen(
    ["uvicorn", "backend.asgi:application", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    cwd=ROOT
)
time.sleep(3)

# Tunnel backend
print("Tunneling backend...")
backend_url = None
def _get_backend():
    global backend_url
    backend_url = _capture_tunnel_url(8000)
threading.Thread(target=_get_backend, daemon=True).start()
while not backend_url:
    time.sleep(1)
print(f"Backend: {backend_url}")

# Build frontend
print("Building frontend...")
build = subprocess.run(
    ["npm", "run", "build"],
    env={**os.environ, "NEXT_PUBLIC_BACKEND_URL": backend_url},
    capture_output=True, text=True,
    cwd=f"{ROOT}/frontend"
)
if build.returncode != 0:
    print(build.stdout[-3000:]); print(build.stderr[-1000:])
    raise RuntimeError("Frontend build failed.")

# Start frontend
print("Starting frontend...")
subprocess.Popen(
    ["npm", "start"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    cwd=f"{ROOT}/frontend"
)
time.sleep(3)

# Tunnel frontend
print("Tunneling frontend...")
frontend_url = None
def _get_frontend():
    global frontend_url
    frontend_url = _capture_tunnel_url(3000)
threading.Thread(target=_get_frontend, daemon=True).start()
while not frontend_url:
    time.sleep(1)

print()
print("=" * 50)
print(f"  Open: {frontend_url}")
print("=" * 50)